# Building a Science Fiction Corpus from HathiTrust Extracted Features

This notebook builds a corpus of science fiction volumes using the [HathiTrust Extracted Features (EF) API](https://htrc.stoplight.io/docs/ef-api/). The EF dataset contains page-level word frequencies, part-of-speech tags, and structural metadata for 18.7 million volumes.

We use Thompson & Mimno's expanded workset of 5,811 speculative fiction volumes from [sf-in-hathitrust](https://github.com/laurejt/sf-in-hathitrust), which matched titles from [Worlds Without End](https://worldswithoutend.com/) and [ISFDB](https://isfdb.org/) against the HathiTrust catalog. This supersedes their original 1,206-volume workset from COLING 2018.

**HTRC is being suspended at end of December 2026.** The EF API is live as of March 2026, but plan accordingly.

## What this notebook does

1. Loads the sci-fi volume ID list (5,811 volumes, 2,235 works, 1,201 authors)
2. Explores the workset metadata (author distribution, publication year, cross-references)
3. Fetches metadata for each volume from the EF API
4. Downloads page-level Extracted Features for a sample of volumes
5. Saves everything locally for analysis in notebook 02

In [ ]:
import requests
import pandas as pd
import json
import time
import os
from pathlib import Path

## 1. Load the Science Fiction Workset

The expanded Thompson & Mimno workset contains 5,811 HathiTrust volume IDs representing 2,235 distinct works by 1,201 authors (20th-century English-language speculative fiction, 1900-2010). Built by matching Worlds Without End + computational text similarity against the HathiTrust catalog.

Fields: `htid`, `HTRecord`, `WWEnd ID`, `ISFDB ID`, `Title`, `Author`, `Year`

In [ ]:
scifi = pd.read_csv('data/sf-hathitrust-volumes.tsv', sep='\t')
print(f"Volumes: {len(scifi)}")
print(f"Unique works (by WWEnd ID): {scifi['WWEnd ID'].nunique()}")
print(f"Unique authors: {scifi['Author'].nunique()}")
print(f"Year range: {scifi['Year'].min()}-{scifi['Year'].max()}")
scifi.head(10)

In [ ]:
# Most represented authors
scifi['Author'].value_counts().head(20)

## 2. Fetch Metadata from the EF API

The EF API endpoint is `https://data.htrc.illinois.edu/ef-api/volumes/{htid}?fields=metadata`. Volume IDs with special characters (`:`, `/`) need encoding: `:` becomes `+`, `/` becomes `=`.

In [ ]:
EF_API = "https://data.htrc.illinois.edu/ef-api/volumes"

def clean_htid(htid):
    """Encode HathiTrust ID for API use."""
    return htid.replace(':', '+').replace('/', '=')

def fetch_metadata(htid):
    """Fetch volume metadata from the EF API."""
    url = f"{EF_API}/{clean_htid(htid)}?fields=metadata"
    resp = requests.get(url, headers={'Accept': 'application/json'})
    if resp.status_code == 200:
        return resp.json().get('data', {}).get('metadata', {})
    return None

In [ ]:
# Test with a single volume
test_meta = fetch_metadata(scifi.iloc[0]['htid'])
print(json.dumps(test_meta, indent=2))

In [ ]:
# Fetch metadata for all volumes
# This takes ~50 minutes for 5,811 volumes at ~2 req/sec
# Uses cached results if available

METADATA_CACHE = 'data/scifi_metadata.json'

if os.path.exists(METADATA_CACHE):
    print(f"Loading cached metadata from {METADATA_CACHE}")
    with open(METADATA_CACHE) as f:
        all_metadata = json.load(f)
else:
    all_metadata = {}

# Fetch any missing IDs
missing = [htid for htid in scifi['htid'] if htid not in all_metadata]
print(f"Cached: {len(all_metadata)}, Missing: {len(missing)}")

if missing:
    errors = []
    for i, htid in enumerate(missing):
        meta = fetch_metadata(htid)
        if meta:
            all_metadata[htid] = meta
        else:
            errors.append(htid)
        
        if (i + 1) % 100 == 0:
            print(f"  {i + 1}/{len(missing)} fetched ({len(errors)} errors)")
            # Save checkpoint
            with open(METADATA_CACHE, 'w') as f:
                json.dump(all_metadata, f)
        
        time.sleep(0.5)
    
    # Final save
    with open(METADATA_CACHE, 'w') as f:
        json.dump(all_metadata, f)
    
    print(f"\nNewly fetched: {len(missing) - len(errors)}, Errors: {len(errors)}")
    if errors:
        print(f"Failed IDs: {errors[:10]}{'...' if len(errors) > 10 else ''}")

print(f"Total metadata records: {len(all_metadata)}")

## 3. Build the Metadata DataFrame

In [ ]:
def parse_metadata(htid, meta):
    """Extract key fields from EF API metadata."""
    contrib = meta.get('contributor', {})
    if isinstance(contrib, list):
        author = '; '.join(c.get('name', '') for c in contrib)
    else:
        author = contrib.get('name', '')
    
    return {
        'htid': htid,
        'title': meta.get('title', ''),
        'author': author,
        'pub_date': meta.get('pubDate', ''),
        'language': meta.get('language', ''),
        'publisher': meta.get('publisher', {}).get('name', '') if isinstance(meta.get('publisher'), dict) else '',
        'pub_place': meta.get('pubPlace', {}).get('name', '') if isinstance(meta.get('pubPlace'), dict) else '',
        'access_rights': meta.get('accessRights', ''),
        'lcc': meta.get('lcc', ''),
        'category': meta.get('category', ''),
        'genre': meta.get('genre', ''),
        'oclc': meta.get('oclc', ''),
        'source': meta.get('sourceInstitution', {}).get('name', '') if isinstance(meta.get('sourceInstitution'), dict) else '',
    }

records = [parse_metadata(htid, meta) for htid, meta in all_metadata.items()]
corpus_df = pd.DataFrame(records)
print(f"Corpus size: {len(corpus_df)} volumes")
corpus_df.head()

In [ ]:
# Publication date distribution
corpus_df['pub_date'] = pd.to_numeric(corpus_df['pub_date'], errors='coerce')
corpus_df['pub_date'].describe()

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 4))
corpus_df['pub_date'].dropna().astype(int).hist(bins=50, ax=ax, color='steelblue', edgecolor='white')
ax.set_xlabel('Publication Year')
ax.set_ylabel('Number of Volumes')
ax.set_title('Science Fiction Corpus: Publication Date Distribution')
plt.tight_layout()
plt.show()

In [ ]:
# Access rights breakdown
# 'pd' = public domain, 'ic' = in copyright
corpus_df['access_rights'].value_counts()

In [ ]:
# Top publishers
corpus_df['publisher'].value_counts().head(15)

In [ ]:
# Top authors
corpus_df['author'].value_counts().head(20)

## 4. Download Extracted Features for a Sample

Full features include page-level token counts with POS tags. Each volume response can be large (1-10 MB), so we start with a sample. The EF data is available regardless of copyright status -- it's non-consumptive (word frequencies, not full text).

In [ ]:
def fetch_volume_features(htid):
    """Fetch full volume data (metadata + page features) from the EF API."""
    url = f"{EF_API}/{clean_htid(htid)}"
    resp = requests.get(url, headers={'Accept': 'application/json'})
    if resp.status_code == 200:
        return resp.json().get('data', {})
    return None

def extract_page_tokens(volume_data):
    """Extract token counts per page from EF volume data."""
    rows = []
    pages = volume_data.get('features', {}).get('pages', [])
    
    for page in pages:
        if not page:
            continue
        seq = page.get('seq', '')
        for section in ['header', 'body', 'footer']:
            sec_data = page.get(section)
            if not sec_data:
                continue
            token_pos = sec_data.get('tokenPosCount', {})
            for token, pos_counts in token_pos.items():
                for pos, count in pos_counts.items():
                    rows.append({
                        'page': seq,
                        'section': section,
                        'token': token,
                        'pos': pos,
                        'count': count
                    })
    
    return pd.DataFrame(rows)

In [ ]:
# Hand-picked sample of well-known sci-fi across eras and styles
# These IDs are verified against the Thompson & Mimno worksets

SAMPLE_HTIDS = [
    'coo.31924001398258',           # Huxley - Island (1972)
    'dul1.ark:/13960/t5gb2ss8d',    # Wells - First Men in the Moon (1901)
    'hvd.32044050819549',           # London - Iron Heel (1917)
    'hvd.hn2vzk',                   # Verne - 20,000 Leagues (1922)
    'inu.30000094835844',           # Asimov - Robot Dreams (2004)
    'mdp.39015002701798',           # Clarke - Sands of Mars (1952)
    'mdp.39015000608953',           # Heinlein - Past Through Tomorrow (1967)
    'inu.39000002436611',           # Bradbury - Illustrated Man (1978)
    'mdp.39015000146798',           # Le Guin - Wind's Twelve Quarters (1975)
    'mdp.39015001151276',           # Dick - Handful of Darkness (1978)
    'mdp.39015011054155',           # Bester - Demolished Man (1978)
    'mdp.39015003465583',           # Ballard - Vermilion Sands (1973)
    'mdp.39015046412311',           # Orwell - 1984 (1949)
    'mdp.39015001537060',           # Herbert - Hellstrom's Hive (1973)
    'inu.30000036612392',           # Banks - Consider Phlebas (1991)
    'mdp.39015020677442',           # Gibson - Burning Chrome (1986)
    'pst.000013408435',             # Atwood - Handmaid's Tale (1986)
    'mdp.39015020728666',           # Butler - Kindred (1988)
    'inu.30000009621628',           # Lem - Memoirs Space Traveller (1991)
    'mdp.39015001577363',           # Zelazny - Eye of Cat (1982)
    'mdp.39015060998203',           # Shelley - Frankenstein (1999 ed.)
    'nyp.33433075746960',           # Shelley - The Last Man (1833)
]

SAMPLE_DIR = Path('data/ef_samples')
SAMPLE_DIR.mkdir(exist_ok=True)

# Show what we'll download
in_expanded = scifi[scifi['htid'].isin(SAMPLE_HTIDS)]
print(f"Sample: {len(SAMPLE_HTIDS)} volumes ({len(in_expanded)} found in expanded workset)")
for _, row in in_expanded.iterrows():
    print(f"  {row['Author'][:30]:30s} | {row['Title'][:50]} ({row['Year']})")

In [ ]:
# Download features for sample volumes
# Each takes 2-5 seconds depending on volume size

for htid in SAMPLE_HTIDS:
    safe_id = htid.replace(':', '_').replace('/', '_')
    cache_path = SAMPLE_DIR / f"{safe_id}.json"
    
    if cache_path.exists():
        print(f"  cached: {htid[:30]}")
        continue
    
    print(f"  fetching: {htid[:30]}...", end='')
    vol_data = fetch_volume_features(htid)
    
    if vol_data:
        with open(cache_path, 'w') as f:
            json.dump(vol_data, f)
        pages = vol_data.get('features', {}).get('pages', [])
        title = vol_data.get('metadata', {}).get('title', '?')[:40]
        print(f" {len(pages)} pages [{title}]")
    else:
        print(" FAILED")
    
    time.sleep(1)

print(f"\nSample files saved to {SAMPLE_DIR}/")

## 5. Save the Corpus Metadata

Save the full metadata DataFrame for use in the analysis notebook.

In [ ]:
corpus_df.to_csv('data/scifi_corpus_metadata.csv', index=False)
print(f"Saved {len(corpus_df)} records to data/scifi_corpus_metadata.csv")

# Summary
print(f"\nCorpus summary:")
print(f"  Total volumes: {len(corpus_df)}")
print(f"  Date range: {int(corpus_df['pub_date'].min())}-{int(corpus_df['pub_date'].max())}")
print(f"  Unique authors: {corpus_df['author'].nunique()}")
print(f"  Public domain: {(corpus_df['access_rights'] == 'pd').sum()}")
print(f"  In copyright: {(corpus_df['access_rights'] == 'ic').sum()}")
print(f"  Sample volumes downloaded: {len(list(SAMPLE_DIR.glob('*.json')))}")

## Next Steps

- **Notebook 02** analyzes the downloaded features: word frequencies, sentiment arcs, vocabulary richness, and cross-volume comparisons
- To expand the sample, add more volume IDs to `SAMPLE_HTIDS` and re-run the download cell
- The expanded workset (5,811 volumes) can be explored via the `scifi` DataFrame -- filter by author, year, or cross-reference with ISFDB
- The 918 anthology volumes in `data/sf-hathitrust-anthologies.tsv` could support short-story segmentation
- For bulk downloads (hundreds of volumes), use rsync: `rsync -azv data.analytics.hathitrust.org::features-2025.04/ ./ef-data/`